In [ ]:
#This notebook is for general tesing and debugging of the code
#You might notice each block is a module from "modular" folder

In [ ]:
#This is to test extraction and save everything for further testing
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickers = ["AAPL.US","MSFT.US"]
end=datetime.today()
start=end-timedelta(days=2*365)
flag=True
for ticker in tickers:
    url=f"https://stooq.com/q/d/l/?s={ticker}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
    df_temp=(pd.read_csv(url,parse_dates=["Date"])
        .set_index("Date")
        .sort_index())
    df_temp.columns = [f"{ticker} Open", f"{ticker} High", f"{ticker} Low", f"{ticker} Close", f"{ticker} Volume"]
    if flag:
        data=df_temp
        flag =False
    else:
        data=data.join(df_temp)

#Defining the period is one of the most important things and key to everything workin propperly
period=pd.Timedelta("2W")
df=data

In [ ]:
#Testing benchmark.py
def amount_of_periods(period,end,start):
    #from inputs import start_date as start
    #from inputs import end_date as end
    #from main_module import start 
    #from main_module import end
    num_periods = (end - start) / period
    round_num_periods=(end - start)//period
    if num_periods!=round_num_periods:
        round_num_periods=round_num_periods+1
    print(round_num_periods)
    return round_num_periods

def calc_dev_by_period(df,companies, period):
    deviations_by_period=[]
    for company in companies:
        deviation=(df[f"{company} High"] - df[f"{company} Low"]).resample(period).std()
        deviations_by_period.append(deviation)
        print(len(deviations_by_period[0]))
    return deviations_by_period

def calc_vola(df, companies, period,end,start):
    deviations_by_period=calc_dev_by_period(df,companies, period)
    num_periods=amount_of_periods(period,end,start)
    volatility_weight=[]
    for i in range(num_periods):
        volatility_weight.append([])
        desv_sum=0
        for j in range(len(companies)):
            deviations_by_period[j][i]=1/deviations_by_period[j][i]
            desv_sum=desv_sum+deviations_by_period[j][i]
        for j in range(len(companies)):
            volatility_weight[i].append(deviations_by_period[j][i]/desv_sum)
    return volatility_weight

In [ ]:
benchmark=calc_vola(df,tickers,period,end,start)

In [ ]:
#Testing portfolio .py
def portfolio_value(benchmark, df, period, tickers):
    periods=amount_of_periods(period, end, start)
    portfolio=[0]*periods
    for j in range(len(tickers)):
        valores=df[f"{tickers[j]} Close"].resample(period).mean()
        for i in range(periods):
            portfolio[i]=portfolio[i]+benchmark[i][j]*valores[i]
    return portfolio

def portfolio_returns(portfolio,period,end,start):
    periods=amount_of_periods(period,end,start)
    port_return=[0]*periods
    for i in range(periods-1):
        port_return[i]=(portfolio[i+1]/portfolio[i])-1
    return port_return

In [ ]:
port=portfolio_value(calc_vola(df,tickers,period, end, start),df,period,tickers)

In [ ]:
r=portfolio_returns(port,period,end,start)
print(r)